# Diagnostic Tests II: Solutions
### Applied Statistical Data Analysis. Prof. Dr. Kristyna Ters | MSc Finance | FHNW

---
> Complete worked solutions with short interpretations. Compare with your own attempts; the reasoning matters as much as the numbers.

In [ ]:
!pip install yfinance pandas-datareader statsmodels --quiet

import yfinance as yf
import pandas_datareader.data as web
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.diagnostic import linear_reset
from statsmodels.stats.stattools import jarque_bera
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.alpha':0.3, 'font.size':11
})
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'
GREY   = '#4B4B4B'; BLUE = '#0E75FE'; GREEN = '#0B7A3C'
print('✓ Libraries loaded.')

---
# Exercise 1: Which Diagnostic? (Solution)

| # | Tool | Rejection / finding means |
|---|------|---------------------------|
| a | **RESET** | curvature the linear form misses; the betas are biased, transform and re-test |
| b | **Jarque-Bera** | non-normal residuals (kurtosis term); relevant for small samples and tails |
| c | **standardized residuals** | flag the days, name the events, run the dummy robustness check |
| d | **Breusch-Pagan / White** (part I) | heteroskedasticity; use robust standard errors |
| e | **RESET** on the levels model | levels relationships are often nonlinear; consider logs |
| f | **event dummies** | re-estimate with dummies for those days; if the beta barely moves, the claim is refuted |

---
# Exercise 2: The Base Model

Our patient for the whole exercise set: the **Nestlé CAPM** against the SMI.

$$r_{NESN,t} = \beta_0 + \beta_1\, r_{SMI,t} + u_t$$

In [ ]:
STOCK, INDEX = 'NESN.SW', '^SSMI'

px  = yf.download([STOCK, INDEX], start='2020-01-01', end='2024-12-31',
                  auto_adjust=True, progress=False)['Close']
ret = px.pct_change().dropna()
# rename by label, never by position: yfinance orders the Close columns alphabetically
ret = ret.rename(columns={STOCK: 'NESN', INDEX: 'SMI'})[['NESN', 'SMI']]

X_capm = sm.add_constant(ret['SMI'])
capm   = sm.OLS(ret['NESN'], X_capm).fit()
print(f'n = {len(ret)} trading days   beta_hat = {capm.params["SMI"]:.4f}   R² = {capm.rsquared:.3f}')

**Interpretation:** a defensive consumer-staples stock typically carries a beta below one. Keep the fitted model; every following exercise interrogates it.

---
# Exercise 3: RESET on the Return Regression

Run Ramsey RESET (powers up to 3, F-form) on the Nestlé CAPM.

**Written question:** the test will very likely pass. Why is a *pass* informative here, and what does it say about the CAPM in returns?

In [ ]:
reset = linear_reset(capm, power=3, use_f=True)
print(f'RESET on the return CAPM: F = {reset.fvalue:.2f}, p = {reset.pvalue:.3f}')
print('→ passes' if reset.pvalue > 0.05 else '→ rejects')

**Answer:** a pass is a result too: the linear return-on-return specification survives a specification test, so treating beta as a constant slope is defensible in this sample. Return regressions usually pass RESET; LEVEL regressions are where functional form goes wrong, as the next exercise shows.

---
# Exercise 4: RESET on a Levels Regression

Now regress the Nestlé PRICE level on the SMI INDEX level (both from the same download, before computing returns) and run RESET again.

**Written question:** compare with Exercise 3. What lesson about levels versus returns do the two results teach?

In [ ]:
lvl = px.dropna().copy()
# rename by label, never by position: yfinance orders the Close columns alphabetically
lvl = lvl.rename(columns={STOCK: 'NESN_P', INDEX: 'SMI_L'})[['NESN_P', 'SMI_L']]

m_lvl = sm.OLS(lvl['NESN_P'], sm.add_constant(lvl['SMI_L'])).fit()
reset_lvl = linear_reset(m_lvl, power=3, use_f=True)

print(f'Levels regression: R² = {m_lvl.rsquared:.3f}')
print(f'RESET: F = {reset_lvl.fvalue:.1f}, p = {reset_lvl.pvalue:.2e}')
print('→ REJECTS: the linear levels specification is wrong' if reset_lvl.pvalue < 0.05 else '→ passes')

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.scatter(m_lvl.fittedvalues, m_lvl.resid, s=6, color=RED, alpha=0.4)
ax.axhline(0, color='black', lw=1)
ax.set_xlabel('fitted value'); ax.set_ylabel('residual')
ax.set_title('Levels regression: systematic residual structure', fontweight='bold', loc='left')
plt.tight_layout(); plt.show()

**Answer:** the same two assets produce a clean specification in returns and a clearly misspecified one in levels: the R² of the levels regression is high, yet RESET rejects and the residual plot shows long systematic swings. High R² is not a specification test. Levels of trending series produce curved, drifting relationships; returns are the safer object for linear models. (The full treatment of trending series follows in the time-series chapters.)

---
# Exercise 5: Jarque-Bera by Hand

Back to the return CAPM from Exercise 2. Compute skewness S and kurtosis K of the residuals, plug them into

$$JB = n\left[\frac{S^2}{6} + \frac{(K-3)^2}{24}\right],$$

and decide against the χ²(2) critical value 5.99.

**Written question:** which of the two terms dominates, and what feature of return data does that reflect?

In [ ]:
u = capm.resid
S = stats.skew(u)
K = stats.kurtosis(u, fisher=False)
n = len(u)

term_S = S**2/6
term_K = (K - 3)**2/24
JB = n * (term_S + term_K)

print(f'S = {S:.2f}  K = {K:.1f}')
print(f'skewness term {term_S:.4f}  vs  kurtosis term {term_K:.3f}')
print(f'JB = {JB:.0f}  (crit 5.99)  → {"REJECT" if JB > 5.99 else "do not reject"}')

**Answer:** the kurtosis term dominates by orders of magnitude. That reflects the fat tails of daily returns, a stylized fact from the very first chapter: extreme days occur far more often than the normal distribution admits, while the distribution is roughly symmetric.

---
# Exercise 6: Confirm and Visualise

Confirm your hand computation with `jarque_bera`, and produce the two standard pictures: histogram of standardized residuals against the normal density, and a QQ-plot.

In [ ]:
jb, jb_p, skew, kurt = jarque_bera(capm.resid)
print(f'jarque_bera: JB = {jb:.0f}, p = {jb_p:.3f}, S = {skew:.2f}, K = {kurt:.1f}')

z = capm.resid / capm.resid.std()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(z, bins=70, density=True, color=YELLOW, edgecolor=GREY, lw=0.3)
xx = np.linspace(-6, 6, 400)
axes[0].plot(xx, stats.norm.pdf(xx), color='black', lw=2)
axes[0].set_xlim(-6, 6)
axes[0].set_title('Standardized residuals vs. normal', fontweight='bold', loc='left')
stats.probplot(z, dist='norm', plot=axes[1])
axes[1].set_title('QQ-plot', fontweight='bold', loc='left')
plt.tight_layout(); plt.show()

**Reading:** peaked centre, heavy tails, and QQ-points leaving the line at both ends. The same signature as for Apple in the lecture: non-normality of daily return residuals is the rule.

---
# Exercise 7: Count and Identify the Outliers

Standardize the residuals, count the days with $|z| > 3$, compare with the count normality predicts, and list the five most extreme days with their dates.

**Written question:** which events do the top dates correspond to? (Think 2020 and, for Swiss stocks, March 2023.)

In [ ]:
z = capm.resid / capm.resid.std()
flagged = z[np.abs(z) > 3]
expected = 2 * (1 - stats.norm.cdf(3)) * len(z)

print(f'observed |z| > 3: {len(flagged)} days   expected under normality: {expected:.1f}')
top5 = z.reindex(z.abs().sort_values(ascending=False).index).head(5)
print('\nMost extreme days:')
print(top5.round(2))

**Answer:** typically several times the predicted count, and the top dates cluster in identifiable episodes: the COVID crash days of March 2020 and, for Swiss stocks, the Credit Suisse takeover turbulence of March 2023. Extreme residuals cluster around nameable events, which is exactly what legitimises the event-dummy treatment.

---
# Exercise 8: The Event-Dummy Robustness Check

Take the three most extreme days you just listed and look up **what happened on each of them**. Then add one 0/1 dummy for every day whose date you can tie to a nameable event, fix that list, and re-estimate with HC1 standard errors. Compare beta, SE and JB with the base model.

**Written question:** state in one sentence what the comparison tells you about the Nestlé beta.

In [ ]:
# The honesty rule, applied. The event days are fixed FIRST, from a named calendar,
# and only then do we look at the residuals. For a Swiss stock the natural calendar is
# the COVID crash week of March 2020 plus the Credit Suisse takeover weekend of 2023.
NAMED_EVENTS = {
    '2020-03-12': 'COVID crash, worst SMI day of the sample',
    '2020-03-16': 'COVID crash, circuit breaker day',
    '2023-03-20': 'Credit Suisse takeover announced over the weekend',
}
events = [pd.Timestamp(d) for d in NAMED_EVENTS if pd.Timestamp(d) in ret.index]
print('Pre-specified event days in the sample:')
rank = z.abs().rank(ascending=False).astype(int)
for d in events:
    print(f'  {d.date()}  {NAMED_EVENTS[d.date().isoformat()]:<50} '
          f'|z| = {abs(z.loc[d]):5.2f}, rank {rank.loc[d]} of {len(z)}')
if not events:
    raise ValueError('none of the named event days is in this sample: pick events '
                     'from the sample period before running the regression')

X_d = ret[['SMI']].copy()
for d in events:
    X_d[f'D_{d.date()}'] = (ret.index == d).astype(float)

capm_hc = sm.OLS(ret['NESN'], X_capm).fit(cov_type='HC1')
capm_d  = sm.OLS(ret['NESN'], sm.add_constant(X_d)).fit(cov_type='HC1')
jb0, _, _, _ = jarque_bera(capm.resid)
jb1, _, _, _ = jarque_bera(capm_d.resid)

cmp = pd.DataFrame({
    'beta_hat': [capm_hc.params['SMI'], capm_d.params['SMI']],
    'SE (HC1)': [capm_hc.bse['SMI'],    capm_d.bse['SMI']],
    'JB':       [round(jb0),            round(jb1)],
}, index=['no dummies', '3 event dummies']).round(4)
print(cmp)


**Answer (one sentence):** the beta barely moves, so Nestlé's estimated systematic risk is not an artefact of a handful of crisis days, and the tightened SE plus the still-rejecting JB confirm that dummies absorb events without manufacturing normality.

---
# Exercise 9: The Honesty Rule, Demonstrated

Now do what one should NOT do: add dummies for EVERY day with $|z| > 2$ and re-estimate. Report how many dummies that takes, the new R², and JB.

**Written question:** the fit improves and JB may finally pass. Why is this bad practice anyway? Give the statistical and the economic argument.

In [ ]:
days2 = z[np.abs(z) > 2].index
X_many = ret[['SMI']].copy()
for d in days2:
    X_many[f'D_{d.date()}'] = (ret.index == d).astype(float)

capm_many = sm.OLS(ret['NESN'], sm.add_constant(X_many)).fit()
jb_many, jb_many_p, _, _ = jarque_bera(capm_many.resid)

print(f'dummies added: {len(days2)}')
print(f'R²: {capm.rsquared:.3f} → {capm_many.rsquared:.3f}')
print(f'JB: {round(jb0)} → {round(jb_many)}  (p = {jb_many_p:.3f})')

**Answer.** Statistically: the dummies are chosen by looking at the residuals, so every conventional p-value in the model is invalidated (data mining), and each dummy mechanically zeroes a residual, so the improvement in fit and in JB is guaranteed rather than informative. Economically: those days ARE the risk. A risk model that dummies away every turbulent day describes a calm world that does not exist, and it will understate exactly the tail risk it is supposed to measure. Dummies are for a handful of nameable events, chosen before looking at significance.

---
# Exercise 10: The Full Verdict (Solution Sketch)

A model answer in six sentences. The return-form of the CAPM passes RESET, so the linear specification is defensible; the levels version fails it, confirming that returns, not levels, are the right object for this model. The residuals are clearly non-normal with a dominating kurtosis term, in line with the fat-tails stylized fact. With more than a thousand observations the central limit theorem protects the t-statistics, so significance statements remain trustworthy, though tail-sensitive applications should not rely on normality. A handful of outlier days cluster around the COVID crash and the March 2023 banking turbulence. An event-dummy robustness check moves the beta only marginally, so the reported beta is not an artefact of those days. I would therefore report the return-CAPM beta with robust standard errors, mention the dummy robustness check in a footnote, and refrain from any claim that relies on normally distributed errors in the tails.

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*